<a href="https://colab.research.google.com/github/charull44/Mental-Health-CheckIn-Dashboard/blob/main/Mental_Health_CheckIn_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install streamlit plotly pyngrok pandas

In [6]:
!file Suicide_Detection.csv.zip

Suicide_Detection.csv.zip: Zip archive data, at least v4.5 to extract, compression method=deflate


In [7]:
!unzip Suicide_Detection.csv.zip

Archive:  Suicide_Detection.csv.zip
  inflating: Suicide_Detection.csv   


In [8]:
import pandas as pd
df = pd.read_csv("Suicide_Detection.csv")

In [9]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px
import os

st.set_page_config(page_title="Mental Health Dashboard", layout="wide")

# ---------- NEON UI ----------
st.markdown("""
<style>
.main {
    background: linear-gradient(135deg, #0f172a, #020617);
    color: white;
}
h1 {
    text-align: center;
    color: #00f5ff;
    text-shadow: 0px 0px 20px #00f5ff;
    font-size: 48px;
}
h2, h3 {
    color: #38bdf8;
}
.stMetric {
    background: rgba(255,255,255,0.05);
    padding: 15px;
    border-radius: 15px;
    box-shadow: 0px 0px 15px rgba(0,255,255,0.3);
}
</style>
""", unsafe_allow_html=True)

# ---------- TITLE ----------
st.markdown("<h1>🧠 Mental Health Dashboard</h1>", unsafe_allow_html=True)
st.markdown("<p style='text-align:center;color:#94a3b8;'>Track mental wellness with interactive insights ✨</p>", unsafe_allow_html=True)

# ---------- LOAD DATA ----------
@st.cache_data
def load_data():
    return pd.read_csv("Suicide_Detection.csv")

df = load_data().dropna()

# ---------- KPI ----------
total = len(df)
suicide_count = len(df[df["class"] == "suicide"])
non_suicide = len(df[df["class"] == "non-suicide"])

col1, col2, col3 = st.columns(3)
col1.metric("📊 Total Records", total)
col2.metric("⚠️ Suicide Posts", suicide_count)
col3.metric("✅ Non-Suicide Posts", non_suicide)

# ---------- PIE CHART ----------
st.subheader("🥧 Class Distribution (Pie Chart)")

pie_fig = px.pie(
    df,
    names="class",
    color="class",
    color_discrete_sequence=["#ff4d6d", "#00f5d4"]
)
pie_fig.update_layout(paper_bgcolor="#020617", font_color="white")

st.plotly_chart(pie_fig, use_container_width=True)

# ---------- BAR CHART (FIXED) ----------
st.subheader("📊 Class Count (Bar Chart)")

bar_data = df["class"].value_counts().reset_index()
bar_data.columns = ["class", "count"]

bar_fig = px.bar(
    bar_data,
    x="class",
    y="count",
    color="class",
    color_discrete_sequence=["#ff4d6d", "#00f5d4"]
)

bar_fig.update_layout(
    plot_bgcolor="#020617",
    paper_bgcolor="#020617",
    font_color="white",
    xaxis_title="Class",
    yaxis_title="Count"
)

st.plotly_chart(bar_fig, use_container_width=True)

# ---------- HISTOGRAM ----------
st.subheader("📊 Text Length Distribution (Histogram)")

df["text_length"] = df["text"].apply(len)

hist_fig = px.histogram(
    df,
    x="text_length",
    color="class",
    nbins=50,
    color_discrete_sequence=["#ff4d6d", "#00f5d4"]
)

hist_fig.update_layout(
    plot_bgcolor="#020617",
    paper_bgcolor="#020617",
    font_color="white"
)

st.plotly_chart(hist_fig, use_container_width=True)

# ---------- SCATTER PLOT ----------
st.subheader("🔵 Scatter Plot (Text Length vs Index)")

df["index"] = df.index

scatter_fig = px.scatter(
    df.sample(min(1000, len(df))),
    x="index",
    y="text_length",
    color="class",
    color_discrete_sequence=["#ff4d6d", "#00f5d4"]
)

scatter_fig.update_layout(
    plot_bgcolor="#020617",
    paper_bgcolor="#020617",
    font_color="white"
)

st.plotly_chart(scatter_fig, use_container_width=True)

# ---------- BOX PLOT ----------
st.subheader("📦 Box Plot Analysis")

box_fig = px.box(
    df,
    x="class",
    y="text_length",
    color="class",
    color_discrete_sequence=["#ff4d6d", "#00f5d4"]
)

box_fig.update_layout(
    plot_bgcolor="#020617",
    paper_bgcolor="#020617",
    font_color="white"
)

st.plotly_chart(box_fig, use_container_width=True)

# ---------- USER TRACKER ----------
st.subheader("📝 Daily Mood Tracker")

if os.path.exists("user_data.csv"):
    user_df = pd.read_csv("user_data.csv")
else:
    user_df = pd.DataFrame(columns=["Date", "Mood", "Stress", "Sleep"])

with st.form("mood_form"):
    date = st.date_input("Select Date")
    mood = st.slider("Mood 😊", 1, 10)
    stress = st.slider("Stress 😖", 1, 10)
    sleep = st.slider("Sleep 😴 (hrs)", 0, 12)

    submitted = st.form_submit_button("Add Entry")

    if submitted:
        new_data = pd.DataFrame([[date, mood, stress, sleep]],
                                columns=["Date", "Mood", "Stress", "Sleep"])
        user_df = pd.concat([user_df, new_data], ignore_index=True)
        user_df.to_csv("user_data.csv", index=False)
        st.success("✨ Entry Added!")

# ---------- USER LINE GRAPH ----------
if not user_df.empty:
    st.subheader("📈 Your Mental Health Trends")

    user_df["Date"] = pd.to_datetime(user_df["Date"])

    line_fig = px.line(
        user_df,
        x="Date",
        y=["Mood", "Stress", "Sleep"],
        markers=True,
        color_discrete_sequence=["#00f5ff", "#ff4d6d", "#ffd60a"]
    )

    line_fig.update_layout(
        plot_bgcolor="#020617",
        paper_bgcolor="#020617",
        font_color="white"
    )

    st.plotly_chart(line_fig, use_container_width=True)

# ---------- INSIGHTS ----------
st.subheader("💡 Insights")

ratio = suicide_count / total

if ratio > 0.5:
    st.error("⚠️ High negative signals detected")
    st.markdown("Try meditation 🧘, proper sleep 😴, and talk to someone 🤝")
else:
    st.success("✅ Balanced mental health pattern")

# ---------- FOOTER ----------
st.markdown("---")
st.markdown("<p style='text-align:center;color:#64748b;'>Made with ❤️ by Charul</p>", unsafe_allow_html=True)

Writing app.py


In [10]:
!ls

app.py	sample_data  Suicide_Detection.csv  Suicide_Detection.csv.zip


In [11]:
!ngrok config add-authtoken 3C20kJ6HaRmq0jS9dPkitxNoaiX_6xFK7QzWweH7T7w9QANb3

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [12]:
!pkill ngrok

In [13]:
from pyngrok import ngrok

!streamlit run app.py &>/dev/null &

ngrok.connect(8501)

<NgrokTunnel: "https://irreducible-unponderous-isaias.ngrok-free.dev" -> "http://localhost:8501">